### Create imbalanced test set

In [3]:
import pandas as pd

gencode = pd.read_csv(r'D:\Splice_FMs_seq_lengths\gencode_multi_seq_length\gencode300.csv')
gtex = pd.read_csv(r'D:\Splice_FMs_seq_lengths\gtex_multi_seq_length\gtex300.csv')

gencode[gencode['CHROM'].isin(['chr20', 'chr21'])]['Splicing_types'].value_counts()

Splicing_types
0    8962
1    8822
2    8666
Name: count, dtype: int64

In [4]:
gtex[gtex['CHROM'].isin(['chr20', 'chr21'])]['Splicing_types'].value_counts()

Splicing_types
1    13776
0    13754
2    13732
Name: count, dtype: int64

In [1]:
from pathlib import Path
import runpy

# Chay script da tao de sinh cac test set imbalanced
script_path = Path("create_imbalanced_test_sets.py")
if not script_path.exists():
    raise FileNotFoundError(f"Khong tim thay script: {script_path.resolve()}")

runpy.run_path(str(script_path), run_name="__main__")

Dang xu ly: D:\Splice_FMs_seq_lengths\gencode_multi_seq_length\gencode300.csv
Tong so mau test (chr20/chr21): 26,450
value_counts Splicing_types (tap test truoc subsample):
Splicing_types
0    8962
1    8822
2    8666
Name: count, dtype: int64

[DONE] gencode300_test_set_1_1_10.csv
Kich thuoc: 10,752 | donor=896, acceptor=896, negative=8,960
value_counts Splicing_types (sau subsample):
Splicing_types
0    8960
1     896
2     896
Name: count, dtype: int64
----------------------------------------------------------------------------------------------------
[DONE] gencode300_test_set_1_1_20.csv
Kich thuoc: 9,856 | donor=448, acceptor=448, negative=8,960
value_counts Splicing_types (sau subsample):
Splicing_types
0    8960
2     448
1     448
Name: count, dtype: int64
----------------------------------------------------------------------------------------------------
[DONE] gencode300_test_set_1_1_50.csv
Kich thuoc: 9,308 | donor=179, acceptor=179, negative=8,950
value_counts Splicing_type

{'__name__': '__main__',
 '__doc__': None,
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'create_imbalanced_test_sets.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, exceptions, and other objects.\n\nNoteworthy: None is the `nil' object; Ellipsis represents `...' in slices.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import__': <function __import__>,
  'abs': <function abs(x, /)>,
  'all': <function all(iterable, /)>,
  'any': <function any(iterable, /)>,
  'ascii': <function ascii(obj, /)>,
  'bin': <function bin(number, /)>,
  'breakpoint': <function breakpoint>,
  'callable': <function callable(obj, /)>,
  'chr': <function chr(i, /)>,
  'compile': <function compile(source, filename, mode, flags=0, dont_inheri

In [2]:
from pathlib import Path

base_dir = Path.cwd()
out_gencode = base_dir / "gencode"
out_gtex = base_dir / "gtex"

print("GENCODE outputs:")
if out_gencode.exists():
    for p in sorted(out_gencode.glob("*.csv")):
        print(" -", p.name)
else:
    print(" - Chua co thu muc gencode")

print("\nGTEx outputs:")
if out_gtex.exists():
    for p in sorted(out_gtex.glob("*.csv")):
        print(" -", p.name)
else:
    print(" - Chua co thu muc gtex")

GENCODE outputs:
 - gencode10000_test_set_1_1_10.csv
 - gencode10000_test_set_1_1_100.csv
 - gencode10000_test_set_1_1_20.csv
 - gencode10000_test_set_1_1_50.csv
 - gencode1000_test_set_1_1_10.csv
 - gencode1000_test_set_1_1_100.csv
 - gencode1000_test_set_1_1_20.csv
 - gencode1000_test_set_1_1_50.csv
 - gencode2000_test_set_1_1_10.csv
 - gencode2000_test_set_1_1_100.csv
 - gencode2000_test_set_1_1_20.csv
 - gencode2000_test_set_1_1_50.csv
 - gencode300_test_set_1_1_10.csv
 - gencode300_test_set_1_1_100.csv
 - gencode300_test_set_1_1_20.csv
 - gencode300_test_set_1_1_50.csv
 - gencode600_test_set_1_1_10.csv
 - gencode600_test_set_1_1_100.csv
 - gencode600_test_set_1_1_20.csv
 - gencode600_test_set_1_1_50.csv

GTEx outputs:
 - gtex10000_test_set_1_1_10.csv
 - gtex10000_test_set_1_1_100.csv
 - gtex10000_test_set_1_1_20.csv
 - gtex10000_test_set_1_1_50.csv
 - gtex1000_test_set_1_1_10.csv
 - gtex1000_test_set_1_1_100.csv
 - gtex1000_test_set_1_1_20.csv
 - gtex1000_test_set_1_1_50.csv
 - gt

### Extract embedding

In [1]:
import os
import sys
import importlib
from pathlib import Path

import pandas as pd
import torch

# Khong dung flash/triton kernels de giam loi runtime
os.environ['FLASH_ATTENTION_DISABLE'] = '1'
os.environ['USE_FLASH_ATTENTION'] = '0'
os.environ['TRITON_DISABLE_LINE_INFO'] = '1'

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))

import config
import models
import splicing_embed_extract
importlib.reload(config)
importlib.reload(models)
importlib.reload(splicing_embed_extract)

from config import EMBEDDINGS_DIR, EMBEDDING_CONFIG, MODELS_CONFIG
from splicing_embed_extract import EmbeddingExtractor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
extractor = EmbeddingExtractor(device=device)

print(f'Project root: {project_root}')
print(f'Device: {device}')
print(f'Embeddings dir: {EMBEDDINGS_DIR}')
print(f'Model families: {list(MODELS_CONFIG.keys())}')

2026-04-12 15:55:31,198 - splicing_embed_extract - INFO - Configured safe attention backend: flash_sdp=False, mem_efficient_sdp=False, math_sdp=True


Project root: d:\Splice_FMs_seq_lengths
Device: cuda
Embeddings dir: d:\Splice_FMs_seq_lengths\data\embeddings
Model families: ['HyenaDNA', 'DNABert', 'NucleotideTransformer']


In [ ]:
from typing import Dict, List
import gc
import json
import time
import inspect
from datetime import datetime

import numpy as np

try:
    import psutil
except Exception:
    psutil = None

WINDOWS = [300, 600, 1000, 2000, 10000]
IMBALANCED_RATIOS = ['1_1_10', '1_1_20', '1_1_50', '1_1_100']
RAW_RATIO_TAG = '1_1_raw'

IMBALANCED_SOURCE_DIRS = {
    'gencode': Path.cwd() / 'gencode',
    'gtex': Path.cwd() / 'gtex',
}

RAW_DEFAULT_DIR = Path.cwd() / 'raw_default_test_set'
RAW_DEFAULT_DIR.mkdir(parents=True, exist_ok=True)

TELEMETRY_DIR = project_root / 'results' / 'pipeline_telemetry'
TELEMETRY_DIR.mkdir(parents=True, exist_ok=True)
extract_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

def _effective_settings(model_name, tokenizer, window_size: int):
    max_length = EMBEDDING_CONFIG.get('max_length', window_size)
    if model_name == 'DNABert':
        tok_max = getattr(tokenizer, 'model_max_length', None)
        if isinstance(tok_max, int) and 0 < tok_max < 100000:
            max_length = min(max_length, tok_max)
        else:
            max_length = min(max_length, 512)
        method = 'center'
        use_fp16 = False
    else:
        method = 'center'
        use_fp16 = EMBEDDING_CONFIG.get('use_fp16', False)
    batch_size = EMBEDDING_CONFIG.get('batch_size_by_window', {}).get(
        window_size, EMBEDDING_CONFIG.get('batch_size', 64)
    )
    return max_length, batch_size, method, use_fp16

def _param_stats(model: torch.nn.Module):
    param_count = sum(p.numel() for p in model.parameters())
    param_mem_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    return int(param_count), float(param_mem_bytes / (1024 ** 2))

def _measure_fm_gflops_per_sample(model, tokenizer, sample_seq: str, max_length: int):
    enc = tokenizer(
        [sample_seq],
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=max_length,
        return_attention_mask=True,
    )
    input_ids = enc['input_ids'].to(device)
    attention_mask = enc['attention_mask'].to(device)

    try:
        model_forward_params = set(inspect.signature(model.forward).parameters.keys())
    except Exception:
        model_forward_params = {'input_ids', 'attention_mask'}

    model_inputs = {'input_ids': input_ids}
    if 'attention_mask' in model_forward_params:
        model_inputs['attention_mask'] = attention_mask

    # 1) Prefer torch profiler FLOPs (full-graph when supported)
    try:
        activities = [torch.profiler.ProfilerActivity.CPU]
        if device == 'cuda' and torch.cuda.is_available():
            activities.append(torch.profiler.ProfilerActivity.CUDA)

        with torch.inference_mode():
            with torch.profiler.profile(
                activities=activities,
                with_flops=True,
                record_shapes=True,
                acc_events=True,
            ) as prof:
                _ = model(**model_inputs)

        total_flops = 0.0
        for evt in prof.key_averages():
            evt_flops = getattr(evt, 'flops', 0)
            if evt_flops is not None:
                total_flops += float(evt_flops)

        if total_flops > 0:
            return total_flops / 1e9, 'measured_profiler'
    except Exception:
        pass

    # 2) Fallback: measured FLOPs from executed nn.Linear layers via hooks
    linear_flops = {'value': 0.0}
    handles = []

    def _linear_hook(module, module_inputs, module_output):
        try:
            in_features = int(module.in_features)
            out_features = int(module.out_features)

            if isinstance(module_output, tuple):
                out_tensor = module_output[0]
            else:
                out_tensor = module_output

            if not torch.is_tensor(out_tensor):
                return

            vectors = out_tensor.numel() / max(1, out_features)
            linear_flops['value'] += vectors * ((2.0 * in_features * out_features) + out_features)
        except Exception:
            return

    for mod in model.modules():
        if isinstance(mod, torch.nn.Linear):
            handles.append(mod.register_forward_hook(_linear_hook))

    try:
        with torch.inference_mode():
            _ = model(**model_inputs)
    finally:
        for h in handles:
            h.remove()

    if linear_flops['value'] > 0:
        return linear_flops['value'] / 1e9, 'measured_linear_hooks'

    raise RuntimeError('Khong do duoc FM GFLOPs (profiler + linear hooks deu that bai).')

def _build_raw_default_csv(window_size: int, source_name: str):
    if source_name == 'gencode':
        source_file = project_root / 'gencode_multi_seq_length' / f'gencode{window_size}.csv'
    else:
        source_file = project_root / 'gtex_multi_seq_length' / f'gtex{window_size}.csv'

    if not source_file.exists():
        raise FileNotFoundError(f'Khong tim thay file goc: {source_file}')

    df = pd.read_csv(source_file)
    if 'CHROM' not in df.columns:
        raise ValueError(f'File {source_file} thieu cot CHROM')

    raw_test_df = df[df['CHROM'].isin(['chr20', 'chr21'])].copy()
    out_file = RAW_DEFAULT_DIR / f'{source_name}{window_size}_test_set_{RAW_RATIO_TAG}.csv'
    raw_test_df.to_csv(out_file, index=False)
    return out_file

def _extract_for_csv(
    extractor_obj,
    model,
    tokenizer,
    csv_path: Path,
    output_path: Path,
    model_name: str,
    model_id: str,
    window_size: int,
    source_name: str,
    ratio: str,
    fm_param_count: int,
    fm_param_memory_mb: float,
    fm_gflops_per_sample: float,
    fm_gflops_method: str,
    telemetry_rows: list,
    force_reextract: bool = True,
):
    df = pd.read_csv(csv_path)
    if 'sequence' not in df.columns or 'Splicing_types' not in df.columns:
        raise ValueError(f'File {csv_path} thieu cot sequence hoac Splicing_types')

    if force_reextract and output_path.exists():
        output_path.unlink()

    sequences = df['sequence'].values
    labels = df['Splicing_types'].values
    max_length, batch_size, method, use_fp16 = _effective_settings(model_name, tokenizer, window_size)

    cpu_before_mb = np.nan
    cpu_after_mb = np.nan
    if psutil is not None:
        proc = psutil.Process()
        cpu_before_mb = proc.memory_info().rss / (1024 ** 2)

    if device == 'cuda' and torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    start_time = time.perf_counter()
    ok = extractor_obj._extract_with_fallback(
        sequences=sequences,
        labels=labels,
        model=model,
        tokenizer=tokenizer,
        output_file=output_path,
        max_length=max_length,
        batch_size=batch_size,
        method=method,
        use_fp16_current=use_fp16,
    )
    elapsed = float(time.perf_counter() - start_time)

    if not ok:
        raise RuntimeError(f'Extract failed for {csv_path.name}')

    gpu_peak_mb = np.nan
    if device == 'cuda' and torch.cuda.is_available():
        gpu_peak_mb = float(torch.cuda.max_memory_allocated() / (1024 ** 2))

    if psutil is not None:
        proc = psutil.Process()
        cpu_after_mb = proc.memory_info().rss / (1024 ** 2)

    telemetry_rows.append({
        'timestamp': extract_timestamp,
        'stage': 'extract_embedding',
        'family': model_name,
        'model_id': model_id,
        'model_variant': model_id,
        'window_size': int(window_size),
        'dataset': source_name,
        'ratio': ratio,
        'input_csv': str(csv_path),
        'output_embedding': str(output_path),
        'n_samples': int(len(df)),
        'running_time_seconds': elapsed,
        'param_count': int(fm_param_count),
        'model_param_memory_mb': float(fm_param_memory_mb),
        'gpu_peak_memory_mb': float(gpu_peak_mb),
        'cpu_ram_delta_mb': float(cpu_after_mb - cpu_before_mb) if psutil is not None else np.nan,
        'gflops_per_sample': float(fm_gflops_per_sample),
        'gflops_total': float(fm_gflops_per_sample * len(df)),
        'gflops_method': fm_gflops_method,
        'batch_size': int(batch_size),
        'max_length': int(max_length),
        'pooling_method': method,
    })

extract_telemetry_rows = []

for window_size in WINDOWS:
    print('\n' + '=' * 100)
    print(f'WINDOW {window_size}')

    raw_csv_by_source = {}
    for source_name in ['gencode', 'gtex']:
        raw_csv = _build_raw_default_csv(window_size, source_name)
        raw_csv_by_source[source_name] = raw_csv
        print(f'[RAW ] {raw_csv.name}')

    for model_name, model_cfg in MODELS_CONFIG.items():
        for model_id in model_cfg.get('model_ids', []):
            skip_reason = config.get_model_window_skip_reason(model_name, model_id, window_size)
            if skip_reason is not None:
                print(f'[SKIP] {skip_reason}')
                continue

            combo_name = f'{model_name}_{model_id}'
            model_dir = EMBEDDINGS_DIR / str(window_size) / combo_name
            model_dir.mkdir(parents=True, exist_ok=True)

            print('\n' + '-' * 100)
            print(f'Loading model: {combo_name}')
            model, tokenizer = extractor.model_loader.load_model_by_name(model_name, model_id)

            fm_param_count, fm_param_memory_mb = _param_stats(model)
            probe_seq = 'ACGT' * max(50, min(window_size // 4, 500))
            fm_gflops_per_sample, fm_gflops_method = _measure_fm_gflops_per_sample(
                model=model,
                tokenizer=tokenizer,
                sample_seq=probe_seq,
                max_length=window_size,
            )
            print(
                f'[FM] params={fm_param_count:,} | param_mem_mb={fm_param_memory_mb:.2f} | '
                f'gflops_per_sample={fm_gflops_per_sample:.4f} ({fm_gflops_method})'
            )

            for source_name in ['gencode', 'gtex']:
                raw_csv = raw_csv_by_source[source_name]
                if source_name == 'gencode':
                    raw_out = model_dir / 'test_embeddings.pt'
                else:
                    raw_out = model_dir / 'gtex_test_embeddings.pt'
                print(f'[RUN ] {raw_csv.name} -> {raw_out.name}')
                _extract_for_csv(
                    extractor_obj=extractor,
                    model=model,
                    tokenizer=tokenizer,
                    csv_path=raw_csv,
                    output_path=raw_out,
                    model_name=model_name,
                    model_id=model_id,
                    window_size=window_size,
                    source_name=source_name,
                    ratio=RAW_RATIO_TAG,
                    fm_param_count=fm_param_count,
                    fm_param_memory_mb=fm_param_memory_mb,
                    fm_gflops_per_sample=fm_gflops_per_sample,
                    fm_gflops_method=fm_gflops_method,
                    telemetry_rows=extract_telemetry_rows,
                    force_reextract=True,
                )

            for source_name, source_dir in IMBALANCED_SOURCE_DIRS.items():
                for ratio in IMBALANCED_RATIOS:
                    csv_file = source_dir / f'{source_name}{window_size}_test_set_{ratio}.csv'
                    if not csv_file.exists():
                        print(f'[SKIP] Missing: {csv_file}')
                        continue

                    if source_name == 'gencode':
                        out_file = model_dir / f'test_{ratio}_embeddings.pt'
                    else:
                        out_file = model_dir / f'gtex_test_{ratio}_embeddings.pt'

                    print(f'[RUN ] {csv_file.name} -> {out_file.name}')
                    _extract_for_csv(
                        extractor_obj=extractor,
                        model=model,
                        tokenizer=tokenizer,
                        csv_path=csv_file,
                        output_path=out_file,
                        model_name=model_name,
                        model_id=model_id,
                        window_size=window_size,
                        source_name=source_name,
                        ratio=ratio,
                        fm_param_count=fm_param_count,
                        fm_param_memory_mb=fm_param_memory_mb,
                        fm_gflops_per_sample=fm_gflops_per_sample,
                        fm_gflops_method=fm_gflops_method,
                        telemetry_rows=extract_telemetry_rows,
                        force_reextract=True,
                    )

            del model
            del tokenizer
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

if not extract_telemetry_rows:
    raise RuntimeError('Khong tao duoc telemetry extract nao. Kiem tra input CSV.')

extract_df = pd.DataFrame(extract_telemetry_rows)
extract_long_csv = TELEMETRY_DIR / f'extract_telemetry_long_{extract_timestamp}.csv'
extract_long_json = TELEMETRY_DIR / f'extract_telemetry_long_{extract_timestamp}.json'
extract_df.to_csv(extract_long_csv, index=False)
extract_df.to_json(extract_long_json, orient='records', indent=2)

extract_window_df = (
    extract_df.groupby(['family', 'model_id', 'model_variant', 'window_size'], as_index=False)
    .agg({
        'running_time_seconds': 'sum',
        'n_samples': 'sum',
        'param_count': 'first',
        'model_param_memory_mb': 'first',
        'gpu_peak_memory_mb': 'max',
        'cpu_ram_delta_mb': 'sum',
        'gflops_per_sample': 'first',
        'gflops_total': 'sum',
        'gflops_method': 'first',
    })
    .rename(columns={
        'running_time_seconds': 'fm_running_time_seconds',
        'param_count': 'fm_param_count',
        'model_param_memory_mb': 'fm_model_param_memory_mb',
        'gpu_peak_memory_mb': 'fm_gpu_peak_memory_mb',
        'cpu_ram_delta_mb': 'fm_cpu_ram_delta_mb',
        'gflops_per_sample': 'fm_gflops_per_sample',
        'gflops_total': 'fm_gflops_total',
    })
)

extract_window_csv = TELEMETRY_DIR / f'extract_telemetry_model_window_{extract_timestamp}.csv'
extract_window_json = TELEMETRY_DIR / f'extract_telemetry_model_window_{extract_timestamp}.json'
extract_window_df.to_csv(extract_window_csv, index=False)
extract_window_df.to_json(extract_window_json, orient='records', indent=2)

print('\nDone extracting embeddings (raw + imbalanced) for all MODELS_CONFIG models.')
print(f'[SAVED] {extract_long_csv.name} | rows={len(extract_df)}')
print(f'[SAVED] {extract_long_json.name} | rows={len(extract_df)}')
print(f'[SAVED] {extract_window_csv.name} | rows={len(extract_window_df)}')
print(f'[SAVED] {extract_window_json.name} | rows={len(extract_window_df)}')


WINDOW 300
[RAW ] gencode300_test_set_1_1_raw.csv
[RAW ] gtex300_test_set_1_1_raw.csv

----------------------------------------------------------------------------------------------------
Loading model: HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf


2026-04-12 02:25:39,478 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/LongSafari/hyenadna-small-32k-seqlen-hf/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-12 02:25:39,516 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/LongSafari/hyenadna-small-32k-seqlen-hf/8fe770c78eb13fe33bf81501612faeddf4d6f331/config.json "HTTP/1.1 200 OK"
2026-04-12 02:25:39,786 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/LongSafari/hyenadna-small-32k-seqlen-hf/resolve/main/configuration_hyena.py "HTTP/1.1 307 Temporary Redirect"
2026-04-12 02:25:39,828 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/LongSafari/hyenadna-small-32k-seqlen-hf/8fe770c78eb13fe33bf81501612faeddf4d6f331/configuration_hyena.py "HTTP/1.1 200 OK"
2026-04-12 02:25:40,097 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/LongSafari/hyenadna-small-32k-seqlen-hf/resolve/main/config.json "HTTP/1.1 307 Temporary Redi

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

HyenaDNAModel LOAD REPORT from: LongSafari/hyenadna-small-32k-seqlen-hf
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-04-12 02:25:42,652 - models - INFO - Successfully loaded HyenaDNA model: LongSafari/hyenadna-small-32k-seqlen-hf
2026-04-12 02:25:42,960 - splicing_embed_extract - INFO - Extracting embeddings using 'center' method for 26450 sequences
2026-04-12 02:25:42,960 - splicing_embed_extract - INFO - Batch size: 256, FP16: True


[FM] params=3,277,568 | param_mem_mb=12.50 | gflops_per_sample=1.9500 (measured_profiler)
[RUN ] gencode300_test_set_1_1_raw.csv -> test_embeddings.pt


...

[RUN ] gtex10000_test_set_1_1_50.csv -> gtex_test_1_1_50_embeddings.pt


2026-04-12 12:28:56,622 - splicing_embed_extract - INFO - Extracting embeddings using 'center' method for 14196 sequences
2026-04-12 12:28:56,622 - splicing_embed_extract - INFO - Batch size: 4, FP16: True
Extracting embeddings: 100%|██████████| 3549/3549 [11:14<00:00,  5.26it/s]
2026-04-12 12:40:11,068 - splicing_embed_extract - INFO - ✓ Saved embeddings: d:\Splice_FMs_seq_lengths\data\embeddings\10000\NucleotideTransformer_InstaDeepAI\nucleotide-transformer-v2-250m-multi-species\gtex_test_1_1_50_embeddings.pt
2026-04-12 12:40:11,068 - splicing_embed_extract - INFO -   Shape: embeddings=torch.Size([14196, 768]), labels=torch.Size([14196])


[RUN ] gtex10000_test_set_1_1_100.csv -> gtex_test_1_1_100_embeddings.pt


2026-04-12 12:40:11,862 - splicing_embed_extract - INFO - Extracting embeddings using 'center' method for 13872 sequences
2026-04-12 12:40:11,863 - splicing_embed_extract - INFO - Batch size: 4, FP16: True
Extracting embeddings: 100%|██████████| 3468/3468 [10:59<00:00,  5.26it/s]
2026-04-12 12:51:11,045 - splicing_embed_extract - INFO - ✓ Saved embeddings: d:\Splice_FMs_seq_lengths\data\embeddings\10000\NucleotideTransformer_InstaDeepAI\nucleotide-transformer-v2-250m-multi-species\gtex_test_1_1_100_embeddings.pt
2026-04-12 12:51:11,046 - splicing_embed_extract - INFO -   Shape: embeddings=torch.Size([13872, 768]), labels=torch.Size([13872])



Done extracting embeddings (raw + imbalanced) for all MODELS_CONFIG models.
[SAVED] extract_telemetry_long_20260412_022537.csv | rows=290
[SAVED] extract_telemetry_long_20260412_022537.json | rows=290
[SAVED] extract_telemetry_model_window_20260412_022537.csv | rows=29
[SAVED] extract_telemetry_model_window_20260412_022537.json | rows=29


### Re-extract DNABERT embedding with center token (instead of CLS like before)

In [ ]:
from pathlib import Path
import runpy
import inspect

import pandas as pd
import numpy as np
import torch

# Khoi tao bien mac dinh neu kernel hien tai chua co
globals_ = globals()

if "WINDOWS" not in globals_:
    WINDOWS = [300, 600, 1000, 2000, 10000]
if "IMBALANCED_RATIOS" not in globals_:
    IMBALANCED_RATIOS = ["1_1_10", "1_1_20", "1_1_50", "1_1_100"]
if "IMBALANCED_SOURCE_DIRS" not in globals_:
    IMBALANCED_SOURCE_DIRS = {
        "gencode": Path.cwd() / "gencode",
        "gtex": Path.cwd() / "gtex",
    }
if "RAW_RATIO_TAG" not in globals_:
    RAW_RATIO_TAG = "1_1_raw"
if "TELEMETRY_DIR" not in globals_:
    TELEMETRY_DIR = Path.cwd().parent / "results" / "pipeline_telemetry"
    TELEMETRY_DIR.mkdir(parents=True, exist_ok=True)

if "psutil" not in globals_:
    try:
        import psutil
    except Exception:
        psutil = None

# Dong bo bien DEVICE/device neu chi co 1 trong 2
if "device" not in globals_ and "DEVICE" in globals_:
    try:
        device = DEVICE.type
    except Exception:
        device = "cuda" if torch.cuda.is_available() else "cpu"
if "DEVICE" not in globals_ and "device" in globals_:
    DEVICE = torch.device(device)

# Ham ho tro neu chua ton tai
if "_build_raw_default_csv" not in globals_:
    def _build_raw_default_csv(window_size: int, source_name: str):
        project_root = Path.cwd().parent
        if source_name == "gencode":
            source_file = project_root / "gencode_multi_seq_length" / f"gencode{window_size}.csv"
        else:
            source_file = project_root / "gtex_multi_seq_length" / f"gtex{window_size}.csv"

        if not source_file.exists():
            raise FileNotFoundError(f"Khong tim thay file goc: {source_file}")

        df = pd.read_csv(source_file)
        if "CHROM" not in df.columns:
            raise ValueError(f"File {source_file} thieu cot CHROM")

        raw_default_dir = Path.cwd() / "raw_default_test_set"
        raw_default_dir.mkdir(parents=True, exist_ok=True)

        raw_test_df = df[df["CHROM"].isin(["chr20", "chr21"])].copy()
        out_file = raw_default_dir / f"{source_name}{window_size}_test_set_{RAW_RATIO_TAG}.csv"
        raw_test_df.to_csv(out_file, index=False)
        return out_file

if "_param_stats" not in globals_:
    def _param_stats(model: torch.nn.Module):
        param_count = sum(p.numel() for p in model.parameters())
        param_mem_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
        return int(param_count), float(param_mem_bytes / (1024 ** 2))

if "_measure_fm_gflops_per_sample" not in globals_:
    def _measure_fm_gflops_per_sample(model, tokenizer, sample_seq: str, max_length: int):
        dev = torch.device(device)
        enc = tokenizer(
            [sample_seq],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
            return_attention_mask=True,
        )
        input_ids = enc["input_ids"].to(dev)
        attention_mask = enc["attention_mask"].to(dev)

        try:
            model_forward_params = set(inspect.signature(model.forward).parameters.keys())
        except Exception:
            model_forward_params = {"input_ids", "attention_mask"}

        model_inputs = {"input_ids": input_ids}
        if "attention_mask" in model_forward_params:
            model_inputs["attention_mask"] = attention_mask

        try:
            activities = [torch.profiler.ProfilerActivity.CPU]
            if dev.type == "cuda" and torch.cuda.is_available():
                activities.append(torch.profiler.ProfilerActivity.CUDA)

            with torch.inference_mode():
                with torch.profiler.profile(
                    activities=activities,
                    with_flops=True,
                    record_shapes=True,
                    acc_events=True,
                ) as prof:
                    _ = model(**model_inputs)

            total_flops = 0.0
            for evt in prof.key_averages():
                evt_flops = getattr(evt, "flops", 0)
                if evt_flops is not None:
                    total_flops += float(evt_flops)

            if total_flops > 0:
                return total_flops / 1e9, "measured_profiler"
        except Exception:
            pass

        linear_flops = {"value": 0.0}
        handles = []

        def _linear_hook(module, module_inputs, module_output):
            try:
                in_features = int(module.in_features)
                out_features = int(module.out_features)
                if isinstance(module_output, tuple):
                    out_tensor = module_output[0]
                else:
                    out_tensor = module_output
                if not torch.is_tensor(out_tensor):
                    return
                vectors = out_tensor.numel() / max(1, out_features)
                linear_flops["value"] += vectors * ((2.0 * in_features * out_features) + out_features)
            except Exception:
                return

        for mod in model.modules():
            if isinstance(mod, torch.nn.Linear):
                handles.append(mod.register_forward_hook(_linear_hook))

        try:
            with torch.inference_mode():
                _ = model(**model_inputs)
        finally:
            for h in handles:
                h.remove()

        if linear_flops["value"] > 0:
            return linear_flops["value"] / 1e9, "measured_linear_hooks"

        raise RuntimeError("Khong do duoc FM GFLOPs (profiler + linear hooks deu that bai).")

required_vars = [
    "config",
    "extractor",
    "EMBEDDINGS_DIR",
    "EMBEDDING_CONFIG",
    "MODELS_CONFIG",
    "WINDOWS",
    "IMBALANCED_RATIOS",
    "IMBALANCED_SOURCE_DIRS",
    "RAW_RATIO_TAG",
    "TELEMETRY_DIR",
    "device",
    "psutil",
    "_build_raw_default_csv",
    "_param_stats",
    "_measure_fm_gflops_per_sample",
]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(
        "Thieu bien tu cac cell setup truoc do: " + ", ".join(missing)
    )

script_path = Path.cwd() / "rerun_dnabert_center_extract.py"
if not script_path.exists():
    raise FileNotFoundError(f"Khong tim thay script: {script_path}")

ctx = {
    "config": config,
    "extractor": extractor,
    "EMBEDDINGS_DIR": EMBEDDINGS_DIR,
    "EMBEDDING_CONFIG": EMBEDDING_CONFIG,
    "MODELS_CONFIG": MODELS_CONFIG,
    "WINDOWS": WINDOWS,
    "IMBALANCED_RATIOS": IMBALANCED_RATIOS,
    "IMBALANCED_SOURCE_DIRS": IMBALANCED_SOURCE_DIRS,
    "RAW_RATIO_TAG": RAW_RATIO_TAG,
    "TELEMETRY_DIR": TELEMETRY_DIR,
    "device": device,
    "psutil": psutil,
    "_build_raw_default_csv": _build_raw_default_csv,
    "_param_stats": _param_stats,
    "_measure_fm_gflops_per_sample": _measure_fm_gflops_per_sample,
}

mod = runpy.run_path(str(script_path))
if "run" not in mod:
    raise RuntimeError("Script rerun_dnabert_center_extract.py khong co ham run(context).")

mod["run"](ctx)

print("\nDone DNABERT center re-extract.")

[LOAD] DNABert_zhihan1996/DNABERT-2-117M | window=300 | pooling=center


2026-04-12 15:56:26,697 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/zhihan1996/DNABERT-2-117M/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-12 15:56:26,726 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/zhihan1996/DNABERT-2-117M/7bce263b15377fc15361f52cfab88f8b586abda0/config.json "HTTP/1.1 200 OK"
2026-04-12 15:56:27,024 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/zhihan1996/DNABERT-2-117M/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-12 15:56:27,072 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/zhihan1996/DNABERT-2-117M/7bce263b15377fc15361f52cfab88f8b586abda0/config.json "HTTP/1.1 200 OK"
2026-04-12 15:56:27,348 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/zhihan1996/DNABERT-2-117M/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-12 15:56:27,413 - httpx - INFO - HTTP Request: HEAD https://huggingf

[LOAD] DNABert_zhihan1996/DNABERT-2-117M | window=600 | pooling=center


2026-04-12 15:58:36,016 - splicing_embed_extract - INFO - Extracting embeddings using 'center' method for 26401 sequences
2026-04-12 15:58:36,019 - splicing_embed_extract - INFO - Batch size: 128, FP16: False
Extracting embeddings: 100%|██████████| 207/207 [00:38<00:00,  5.34it/s]
2026-04-12 15:59:14,849 - splicing_embed_extract - INFO - ✓ Saved embeddings: d:\Splice_FMs_seq_lengths\data\embeddings\600\DNABert_zhihan1996\DNABERT-2-117M\test_embeddings.pt
2026-04-12 15:59:14,849 - splicing_embed_extract - INFO -   Shape: embeddings=torch.Size([26401, 768]), labels=torch.Size([26401])
2026-04-12 15:59:15,032 - splicing_embed_extract - INFO - Extracting embeddings using 'center' method for 41261 sequences
2026-04-12 15:59:15,032 - splicing_embed_extract - INFO - Batch size: 128, FP16: False
Extracting embeddings: 100%|██████████| 323/323 [00:55<00:00,  5.82it/s]
2026-04-12 16:00:10,689 - splicing_embed_extract - INFO - ✓ Saved embeddings: d:\Splice_FMs_seq_lengths\data\embeddings\600\DNAB

[LOAD] DNABert_zhihan1996/DNABERT-2-117M | window=1000 | pooling=center


...

[LOAD] DNABert_zhihan1996/DNABERT-2-117M | window=2000 | pooling=center


2026-04-12 16:09:29,099 - splicing_embed_extract - INFO - Extracting embeddings using 'center' method for 26364 sequences
2026-04-12 16:09:29,100 - splicing_embed_extract - INFO - Batch size: 32, FP16: False
Extracting embeddings: 100%|██████████| 824/824 [02:18<00:00,  5.97it/s]
2026-04-12 16:11:47,291 - splicing_embed_extract - INFO - ✓ Saved embeddings: d:\Splice_FMs_seq_lengths\data\embeddings\2000\DNABert_zhihan1996\DNABERT-2-117M\test_embeddings.pt
2026-04-12 16:11:47,293 - splicing_embed_extract - INFO -   Shape: embeddings=torch.Size([26364, 768]), labels=torch.Size([26364])
2026-04-12 16:11:47,714 - splicing_embed_extract - INFO - Extracting embeddings using 'center' method for 41064 sequences
2026-04-12 16:11:47,714 - splicing_embed_extract - INFO - Batch size: 32, FP16: False
Extracting embeddings: 100%|██████████| 1284/1284 [03:30<00:00,  6.11it/s]
2026-04-12 16:15:18,150 - splicing_embed_extract - INFO - ✓ Saved embeddings: d:\Splice_FMs_seq_lengths\data\embeddings\2000\DN

[LOAD] DNABert_zhihan1996/DNABERT-2-117M | window=10000 | pooling=center


2026-04-12 16:24:46,806 - splicing_embed_extract - INFO - Extracting embeddings using 'center' method for 26364 sequences
2026-04-12 16:24:46,806 - splicing_embed_extract - INFO - Batch size: 4, FP16: False
Extracting embeddings: 100%|██████████| 6591/6591 [03:33<00:00, 30.81it/s]
2026-04-12 16:28:20,822 - splicing_embed_extract - INFO - ✓ Saved embeddings: d:\Splice_FMs_seq_lengths\data\embeddings\10000\DNABert_zhihan1996\DNABERT-2-117M\test_embeddings.pt
2026-04-12 16:28:20,823 - splicing_embed_extract - INFO -   Shape: embeddings=torch.Size([26364, 768]), labels=torch.Size([26364])
2026-04-12 16:28:22,759 - splicing_embed_extract - INFO - Extracting embeddings using 'center' method for 41064 sequences
2026-04-12 16:28:22,760 - splicing_embed_extract - INFO - Batch size: 4, FP16: False
Extracting embeddings: 100%|██████████| 10266/10266 [05:31<00:00, 30.99it/s]
2026-04-12 16:33:54,126 - splicing_embed_extract - INFO - ✓ Saved embeddings: d:\Splice_FMs_seq_lengths\data\embeddings\1000

[DONE] DNABERT center re-extract + full telemetry merge
[CHECK] pooling_method counts in DNABERT rows:
pooling_method
center    50
Name: count, dtype: int64
[SAVED] extract_telemetry_long_20260412_164705.csv | rows=290

Done DNABERT center re-extract.


In [4]:
from pathlib import Path

check_windows = [300, 600, 1000, 2000, 10000]
check_ratios = ['1_1_10', '1_1_20', '1_1_50', '1_1_100']
base_embed = Path.cwd().parent / 'data' / 'embeddings'

for ws in check_windows:
    print('\n' + '=' * 100)
    print(f'Check window {ws}')

    for model_name, model_cfg in MODELS_CONFIG.items():
        for model_id in model_cfg.get('model_ids', []):
            if config.get_model_window_skip_reason(model_name, model_id, ws) is not None:
                continue

            model_dir = base_embed / str(ws) / f'{model_name}_{model_id}'
            print(f'  Model: {model_name}_{model_id}')
            print(f'  Dir exists: {model_dir.exists()}')

            for rt in check_ratios:
                f1 = model_dir / f'test_{rt}_embeddings.pt'
                f2 = model_dir / f'gtex_test_{rt}_embeddings.pt'
                print(f'    GENCODE {rt}: {f1.exists()}')
                print(f'    GTEx    {rt}: {f2.exists()}')


Check window 300
  Model: HyenaDNA_LongSafari/hyenadna-small-32k-seqlen-hf
  Dir exists: True
    GENCODE 1_1_10: True
    GTEx    1_1_10: True
    GENCODE 1_1_20: True
    GTEx    1_1_20: True
    GENCODE 1_1_50: True
    GTEx    1_1_50: True
    GENCODE 1_1_100: True
    GTEx    1_1_100: True
  Model: HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf
  Dir exists: True
    GENCODE 1_1_10: True
    GTEx    1_1_10: True
    GENCODE 1_1_20: True
    GTEx    1_1_20: True
    GENCODE 1_1_50: True
    GTEx    1_1_50: True
    GENCODE 1_1_100: True
    GTEx    1_1_100: True
  Model: DNABert_zhihan1996/DNABERT-2-117M
  Dir exists: True
    GENCODE 1_1_10: True
    GTEx    1_1_10: True
    GENCODE 1_1_20: True
    GTEx    1_1_20: True
    GENCODE 1_1_50: True
    GTEx    1_1_50: True
    GENCODE 1_1_100: True
    GTEx    1_1_100: True
  Model: NucleotideTransformer_InstaDeepAI/nucleotide-transformer-500m-human-ref
  Dir exists: True
    GENCODE 1_1_10: True
    GTEx    1_1_10: True
    GENC

### Inference

In [1]:
import json
import os
import re
import sys
import time
import importlib
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch

try:
    import psutil
except Exception:
    psutil = None

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'src'))

import config
importlib.reload(config)
from config import EMBEDDINGS_DIR, SPLICING_RESULTS_DIR
from splicing_classifier import SpliceSiteClassifier
from splicing_metrics import MetricsComputer

EXCLUDE_EXPERIMENTS = {'DNABERT-2-117M_CLS_w300_quick'}
RESULTS_ROOT = Path(SPLICING_RESULTS_DIR)
SUMMARY_DIR = RESULTS_ROOT / 'summaries'
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
TELEMETRY_DIR = project_root / 'results' / 'pipeline_telemetry'
TELEMETRY_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RAW_RATIO_TAG = '1_1_raw'
RATIOS = [RAW_RATIO_TAG, '1_1_10', '1_1_20', '1_1_50', '1_1_100']
INFER_BATCH_SIZE = 1024
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

print(f'Results root: {RESULTS_ROOT}')
print(f'Embeddings root: {EMBEDDINGS_DIR}')
print(f'Telemetry root: {TELEMETRY_DIR}')
print(f'Device: {DEVICE}')
print(f'psutil available: {psutil is not None}')

def _to_serializable(obj):
    if isinstance(obj, dict):
        return {k: _to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_serializable(v) for v in obj]
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

def _normalize_path_str(p):
    try:
        s = str(Path(p))
    except Exception:
        s = str(p)
    s = s.replace('/', '\\')
    return os.path.normcase(os.path.normpath(s))

def _resolve_best_checkpoint(exp_dir: Path, cv_results: dict):
    best_fold_info = cv_results.get('best_fold', {}) or {}
    checkpoint_path = best_fold_info.get('checkpoint_path', None)
    checkpoint_name = best_fold_info.get('checkpoint', None)
    fold_number = best_fold_info.get('fold_number', None)

    if checkpoint_name is None:
        fold_entries = cv_results.get('per_fold_results', {}) or {}
        if not fold_entries:
            return None, None, None

        best_key = None
        best_mcc = -1e9
        for fold_key, fold_val in fold_entries.items():
            mcc = fold_val.get('best_mcc', -1e9)
            if mcc > best_mcc:
                best_mcc = mcc
                best_key = fold_key

        if best_key is None:
            return None, None, None

        checkpoint_name = fold_entries[best_key].get('best_checkpoint', None)
        if checkpoint_name is None:
            try:
                fold_idx = int(best_key.replace('fold_', ''))
                checkpoint_name = f'best_fold{fold_idx + 1}.pt'
            except Exception:
                checkpoint_name = None

        try:
            fold_number = int(best_key.replace('fold_', '')) + 1
        except Exception:
            fold_number = None

    checkpoint_path_obj = None
    if checkpoint_path is not None:
        checkpoint_path_obj = Path(checkpoint_path)
        if not checkpoint_path_obj.exists():
            checkpoint_path_obj = None

    if checkpoint_path_obj is None and checkpoint_name is not None:
        checkpoint_path_obj = exp_dir / 'checkpoints' / checkpoint_name

    if checkpoint_path_obj is None or not checkpoint_path_obj.exists():
        return None, checkpoint_name, fold_number

    return checkpoint_path_obj, checkpoint_name, fold_number

def _parse_experiment(exp_dir: Path):
    family_name = exp_dir.parent.name
    exp_name = exp_dir.name
    m = re.search(r'_w(\d+)', exp_name)
    if m is None:
        return None
    window_size = int(m.group(1))
    model_variant = exp_name.split(f'_w{window_size}', 1)[0]
    embed_dir = EMBEDDINGS_DIR / str(window_size) / family_name / model_variant
    return {
        'family_name': family_name,
        'exp_name': exp_name,
        'window_size': window_size,
        'model_variant': model_variant,
        'embed_dir': embed_dir,
    }

def _list_embedding_targets(embed_dir: Path):
    targets = [
        ('gencode', RAW_RATIO_TAG, embed_dir / 'test_embeddings.pt'),
        ('gtex', RAW_RATIO_TAG, embed_dir / 'gtex_test_embeddings.pt'),
    ]
    for ratio in RATIOS:
        if ratio == RAW_RATIO_TAG:
            continue
        targets.append(('gencode', ratio, embed_dir / f'test_{ratio}_embeddings.pt'))
        targets.append(('gtex', ratio, embed_dir / f'gtex_test_{ratio}_embeddings.pt'))
    return [(dataset, ratio, fp) for dataset, ratio, fp in targets if fp.exists()]

def _classifier_gflops_per_sample(model: torch.nn.Module):
    total_flops = 0.0
    for layer in model.modules():
        if isinstance(layer, torch.nn.Linear):
            in_dim = int(layer.in_features)
            out_dim = int(layer.out_features)
            total_flops += (2.0 * in_dim * out_dim) + out_dim
    return total_flops / 1e9

def _model_param_stats(model: torch.nn.Module):
    param_count = sum(p.numel() for p in model.parameters())
    param_mem_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    return int(param_count), float(param_mem_bytes / (1024 ** 2))

def _run_inference_on_embedding_file(checkpoint_payload, embedding_file, batch_size=1024):
    data = torch.load(embedding_file, map_location='cpu')
    embeddings = data['embeddings']
    labels = data['labels']

    if not isinstance(embeddings, torch.Tensor):
        embeddings = torch.tensor(embeddings, dtype=torch.float32)
    else:
        embeddings = embeddings.float()

    if not isinstance(labels, torch.Tensor):
        labels = torch.tensor(labels, dtype=torch.long)
    else:
        labels = labels.long()

    cpu_before_mb = np.nan
    cpu_after_mb = np.nan
    if psutil is not None:
        proc = psutil.Process(os.getpid())
        cpu_before_mb = proc.memory_info().rss / (1024 ** 2)

    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    model_state = checkpoint_payload.get('model_state_dict', checkpoint_payload)
    embedding_dim = int(checkpoint_payload.get('embedding_dim', embeddings.shape[1]))
    num_classes = int(checkpoint_payload.get('num_classes', 3))

    model = SpliceSiteClassifier(
        embedding_dim=embedding_dim,
        num_classes=num_classes,
        hidden_dims=[512, 256],
        dropout_rate=0.3,
    ).to(DEVICE)
    model.load_state_dict(model_state)
    model.eval()

    param_count, param_mem_mb = _model_param_stats(model)
    gflops_per_sample = _classifier_gflops_per_sample(model)

    all_preds = []
    all_probs = []
    start_time = time.perf_counter()
    with torch.no_grad():
        for start_idx in range(0, len(embeddings), batch_size):
            end_idx = min(start_idx + batch_size, len(embeddings))
            batch_x = embeddings[start_idx:end_idx].to(DEVICE, non_blocking=(DEVICE.type == 'cuda'))
            logits = model(batch_x)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            all_preds.append(preds.cpu())
            all_probs.append(probs.cpu())
    infer_seconds = float(time.perf_counter() - start_time)

    gpu_peak_mb = np.nan
    if DEVICE.type == 'cuda':
        gpu_peak_mb = float(torch.cuda.max_memory_allocated() / (1024 ** 2))

    if psutil is not None:
        proc = psutil.Process(os.getpid())
        cpu_after_mb = proc.memory_info().rss / (1024 ** 2)

    y_true = labels.cpu().numpy()
    y_pred = torch.cat(all_preds, dim=0).numpy()
    y_prob = torch.cat(all_probs, dim=0).numpy()
    metrics, cm = MetricsComputer.compute_metrics(y_true, y_pred, y_prob)

    del model
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    return {
        'num_samples': int(len(y_true)),
        'running_time_seconds': infer_seconds,
        'cpu_ram_delta_mb': float(cpu_after_mb - cpu_before_mb) if psutil is not None else np.nan,
        'gpu_peak_memory_mb': float(gpu_peak_mb),
        'param_count': param_count,
        'model_param_memory_mb': param_mem_mb,
        'gflops_per_sample': float(gflops_per_sample),
        'gflops_total': float(gflops_per_sample * len(y_true)),
        'metrics': {k: float(v) if isinstance(v, (int, float, np.number)) else v for k, v in metrics.items()},
        'confusion_matrix': cm.tolist(),
    }

result_rows = []
legacy_inference_results = {}
result_files = sorted(RESULTS_ROOT.glob('*/*/results.json'))
print(f'Found experiments with results.json: {len(result_files)}')

for rs_file in result_files:
    exp_dir = rs_file.parent
    parsed = _parse_experiment(exp_dir)
    if parsed is None:
        continue

    if parsed['exp_name'] in EXCLUDE_EXPERIMENTS:
        print(f"[SKIP-EXCLUDED] {parsed['exp_name']}")
        continue

    with open(rs_file, 'r', encoding='utf-8') as f:
        cv_results = json.load(f)

    checkpoint_path, checkpoint_name, fold_number = _resolve_best_checkpoint(exp_dir, cv_results)
    if checkpoint_path is None:
        print(f"[SKIP-NOCKPT] {exp_dir}")
        continue

    targets = _list_embedding_targets(parsed['embed_dir'])
    if not targets:
        print(f"[SKIP-NO-EMB] {parsed['embed_dir']}")
        continue

    checkpoint_payload = torch.load(checkpoint_path, map_location='cpu')
    model_infer_total = 0.0

    print(f"[RUN] {parsed['family_name']}/{parsed['exp_name']} | targets={len(targets)}")

    for dataset_name, ratio_name, emb_file in targets:
        infer_out = _run_inference_on_embedding_file(
            checkpoint_payload=checkpoint_payload,
            embedding_file=emb_file,
            batch_size=INFER_BATCH_SIZE,
        )
        model_infer_total += infer_out['running_time_seconds']
        row = {
            'timestamp': timestamp,
            'stage': 'mlp_inference',
            'family': parsed['family_name'],
            'model_id': parsed['model_variant'],
            'model_variant': parsed['model_variant'],
            'experiment': parsed['exp_name'],
            'window_size': parsed['window_size'],
            'dataset': dataset_name,
            'ratio': ratio_name,
            'best_fold_number': fold_number,
            'checkpoint': checkpoint_name,
            'best_checkpoint': str(checkpoint_path),
            'embedding_file': str(emb_file),
            'running_time_seconds': infer_out['running_time_seconds'],
            'model_inference_seconds_total': np.nan,
            'n_samples': infer_out['num_samples'],
            'param_count': infer_out['param_count'],
            'model_param_memory_mb': infer_out['model_param_memory_mb'],
            'gpu_peak_memory_mb': infer_out['gpu_peak_memory_mb'],
            'cpu_ram_delta_mb': infer_out['cpu_ram_delta_mb'],
            'gflops_per_sample': infer_out['gflops_per_sample'],
            'gflops_total': infer_out['gflops_total'],
            'confusion_matrix': json.dumps(infer_out['confusion_matrix']),
            **infer_out['metrics'],
        }
        result_rows.append(row)

    for row in result_rows[::-1]:
        if row['experiment'] == parsed['exp_name']:
            row['model_inference_seconds_total'] = float(model_infer_total)
        else:
            break

    default_gencode = [r for r in result_rows if r['experiment'] == parsed['exp_name'] and r['ratio'] == RAW_RATIO_TAG and r['dataset'] == 'gencode']
    default_gtex = [r for r in result_rows if r['experiment'] == parsed['exp_name'] and r['ratio'] == RAW_RATIO_TAG and r['dataset'] == 'gtex']
    if default_gencode and default_gtex:
        dg = default_gencode[-1]
        dt = default_gtex[-1]
        metric_keys = [k for k in dg.keys() if k.startswith('accuracy') or k.startswith('f1') or k.startswith('mcc') or k.startswith('balanced') or k.startswith('roc_auc') or k.startswith('pr_auc')]
        legacy_inference_results[parsed['exp_name']] = {
            'window_size': parsed['window_size'],
            'model_name': parsed['family_name'].split('_')[0],
            'model_id': parsed['model_variant'],
            'best_fold_number': fold_number,
            'checkpoint': checkpoint_name,
            'checkpoint_path': str(checkpoint_path),
            'model_inference_seconds_total': float(model_infer_total),
            'gencode_test': {
                'num_samples': int(dg['n_samples']),
                'metrics': {k: dg[k] for k in metric_keys if k in dg},
                'confusion_matrix': json.loads(dg['confusion_matrix']),
                'running_time_seconds': float(dg['running_time_seconds']),
            },
            'gtex_test': {
                'num_samples': int(dt['n_samples']),
                'metrics': {k: dt[k] for k in metric_keys if k in dt},
                'confusion_matrix': json.loads(dt['confusion_matrix']),
                'running_time_seconds': float(dt['running_time_seconds']),
            },
        }

if not result_rows:
    raise RuntimeError('Khong co ket qua inference nao duoc tao. Kiem tra embeddings/results folders.')

mlp_df = pd.DataFrame(result_rows)

mlp_long_csv = TELEMETRY_DIR / f'mlp_inference_telemetry_long_{timestamp}.csv'
mlp_long_json = TELEMETRY_DIR / f'mlp_inference_telemetry_long_{timestamp}.json'
mlp_df.to_csv(mlp_long_csv, index=False)
mlp_df.to_json(mlp_long_json, orient='records', indent=2)

extract_long_candidates = sorted(TELEMETRY_DIR.glob('extract_telemetry_long_*.csv'))
if not extract_long_candidates:
    raise RuntimeError('Khong tim thay extract telemetry. Hay chay cell Extract embedding truoc.')
extract_long_path = extract_long_candidates[-1]
extract_df = pd.read_csv(extract_long_path)

# Robust join by exact embedding file path (normalized), because family/model_id naming differs
fm_cols = [
    'running_time_seconds', 'param_count', 'model_param_memory_mb',
    'gpu_peak_memory_mb', 'cpu_ram_delta_mb', 'gflops_per_sample', 'gflops_total', 'gflops_method'
    ]
fm_df = extract_df[['output_embedding'] + fm_cols].copy()
fm_df['embedding_key'] = fm_df['output_embedding'].map(_normalize_path_str)
fm_df = fm_df.rename(columns={
    'running_time_seconds': 'fm_running_time_seconds',
    'param_count': 'fm_param_count',
    'model_param_memory_mb': 'fm_model_param_memory_mb',
    'gpu_peak_memory_mb': 'fm_gpu_peak_memory_mb',
    'cpu_ram_delta_mb': 'fm_cpu_ram_delta_mb',
    'gflops_per_sample': 'fm_gflops_per_sample',
    'gflops_total': 'fm_gflops_total',
    'gflops_method': 'fm_gflops_method',
})
fm_df = fm_df.drop_duplicates(subset=['embedding_key'], keep='last')

mlp_join_df = mlp_df.rename(columns={
    'running_time_seconds': 'mlp_running_time_seconds',
    'param_count': 'mlp_param_count',
    'model_param_memory_mb': 'mlp_model_param_memory_mb',
    'gpu_peak_memory_mb': 'mlp_gpu_peak_memory_mb',
    'cpu_ram_delta_mb': 'mlp_cpu_ram_delta_mb',
    'gflops_per_sample': 'mlp_gflops_per_sample',
    'gflops_total': 'mlp_gflops_total',
}).copy()
mlp_join_df['embedding_key'] = mlp_join_df['embedding_file'].map(_normalize_path_str)

pipeline_df = mlp_join_df.merge(
    fm_df[['embedding_key', 'fm_running_time_seconds', 'fm_param_count', 'fm_model_param_memory_mb', 'fm_gpu_peak_memory_mb', 'fm_cpu_ram_delta_mb', 'fm_gflops_per_sample', 'fm_gflops_total', 'fm_gflops_method']],
    on='embedding_key',
    how='left',
)

matched = int(pipeline_df['fm_running_time_seconds'].notna().sum())
total_rows = int(len(pipeline_df))
print(f'[JOIN] FM matched rows: {matched}/{total_rows}')

pipeline_df['pipeline_running_time_seconds'] = pipeline_df['fm_running_time_seconds'] + pipeline_df['mlp_running_time_seconds']
pipeline_df['pipeline_gflops_total'] = pipeline_df['fm_gflops_total'] + pipeline_df['mlp_gflops_total']
pipeline_df['pipeline_gpu_peak_memory_mb'] = pipeline_df[['fm_gpu_peak_memory_mb', 'mlp_gpu_peak_memory_mb']].max(axis=1)
pipeline_df['pipeline_cpu_ram_delta_mb'] = pipeline_df['fm_cpu_ram_delta_mb'] + pipeline_df['mlp_cpu_ram_delta_mb']
pipeline_df['pipeline_param_count_total'] = pipeline_df['fm_param_count'] + pipeline_df['mlp_param_count']
pipeline_df['pipeline_model_param_memory_mb_total'] = pipeline_df['fm_model_param_memory_mb'] + pipeline_df['mlp_model_param_memory_mb']

ratio_order = {r: i for i, r in enumerate(RATIOS)}
pipeline_df['ratio_order'] = pipeline_df['ratio'].map(ratio_order).fillna(999).astype(int)
pipeline_df = pipeline_df.sort_values(['ratio_order', 'window_size', 'family', 'experiment', 'dataset']).reset_index(drop=True)

pipe_long_csv = TELEMETRY_DIR / f'pipeline_telemetry_long_{timestamp}.csv'
pipe_long_json = TELEMETRY_DIR / f'pipeline_telemetry_long_{timestamp}.json'
pipeline_df.drop(columns=['ratio_order', 'embedding_key']).to_csv(pipe_long_csv, index=False)
pipeline_df.drop(columns=['ratio_order', 'embedding_key']).to_json(pipe_long_json, orient='records', indent=2)

wide_metrics = [
    'balanced_accuracy', 'mcc', 'f1_weighted',
    'pipeline_running_time_seconds', 'pipeline_gflops_total',
    'pipeline_param_count_total', 'pipeline_model_param_memory_mb_total'
    ]
pipeline_wide = pipeline_df.pivot_table(
    index=['family', 'model_variant', 'window_size', 'dataset'],
    columns='ratio',
    values=wide_metrics,
    aggfunc='first',
)
pipeline_wide.columns = [f'{m}_{r}' for m, r in pipeline_wide.columns]
pipeline_wide = pipeline_wide.reset_index()

pipe_wide_csv = TELEMETRY_DIR / f'pipeline_telemetry_wide_{timestamp}.csv'
pipe_wide_json = TELEMETRY_DIR / f'pipeline_telemetry_wide_{timestamp}.json'
pipeline_wide.to_csv(pipe_wide_csv, index=False)
pipeline_wide.to_json(pipe_wide_json, orient='records', indent=2)

all_csv = SUMMARY_DIR / f'bestfold_inference_all_ratios_{timestamp}.csv'
all_json = SUMMARY_DIR / f'bestfold_inference_all_ratios_{timestamp}.json'
pipeline_df.drop(columns=['ratio_order', 'embedding_key']).to_csv(all_csv, index=False)
pipeline_df.drop(columns=['ratio_order', 'embedding_key']).to_json(all_json, orient='records', indent=2)

for ratio_name in RATIOS:
    df_ratio = pipeline_df[pipeline_df['ratio'] == ratio_name].drop(columns=['ratio_order', 'embedding_key'])
    if df_ratio.empty:
        continue
    ratio_csv = SUMMARY_DIR / f'bestfold_inference_ratio_{ratio_name}_{timestamp}.csv'
    ratio_json = SUMMARY_DIR / f'bestfold_inference_ratio_{ratio_name}_{timestamp}.json'
    df_ratio.to_csv(ratio_csv, index=False)
    df_ratio.to_json(ratio_json, orient='records', indent=2)
    print(f'[SAVED] {ratio_csv.name} | rows={len(df_ratio)}')
    print(f'[SAVED] {ratio_json.name} | rows={len(df_ratio)}')

legacy_json = SUMMARY_DIR / f'best_fold_inference_results_{timestamp}.json'
with open(legacy_json, 'w', encoding='utf-8') as f:
    json.dump(_to_serializable(legacy_inference_results), f, indent=2)

results_df = pipeline_df.copy()
print(f'\n[SAVED] {legacy_json.name} | experiments={len(legacy_inference_results)}')
print(f'[SAVED] {mlp_long_csv.name} | rows={len(mlp_df)}')
print(f'[SAVED] {pipe_long_csv.name} | rows={len(pipeline_df)}')
print(f'[SAVED] {pipe_wide_csv.name} | rows={len(pipeline_wide)}')
print(f'[SAVED] {all_csv.name} | rows={len(pipeline_df)}')
results_df.head(10)

Results root: d:\Splice_FMs_seq_lengths\results\classifiers
Embeddings root: d:\Splice_FMs_seq_lengths\data\embeddings
Telemetry root: d:\Splice_FMs_seq_lengths\results\pipeline_telemetry
Device: cuda
psutil available: True
Found experiments with results.json: 30
[SKIP-EXCLUDED] DNABERT-2-117M_CLS_w300_quick
[RUN] DNABert_zhihan1996/DNABERT-2-117M_w1000 | targets=10
[RUN] DNABert_zhihan1996/DNABERT-2-117M_w10000 | targets=10
[RUN] DNABert_zhihan1996/DNABERT-2-117M_w2000 | targets=10
[RUN] DNABert_zhihan1996/DNABERT-2-117M_w300 | targets=10
[RUN] DNABert_zhihan1996/DNABERT-2-117M_w600 | targets=10
[RUN] HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf_w1000 | targets=10
[RUN] HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf_w10000 | targets=10
[RUN] HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf_w2000 | targets=10
[RUN] HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf_w300 | targets=10
[RUN] HyenaDNA_LongSafari/hyenadna-medium-160k-seqlen-hf_w600 | targets=10
[RUN] HyenaDNA_Lo

,timestamp,stage,family,model_id,model_variant,experiment,window_size,dataset,ratio,best_fold_number,...,fm_gflops_per_sample,fm_gflops_total,fm_gflops_method,pipeline_running_time_seconds,pipeline_gflops_total,pipeline_gpu_peak_memory_mb,pipeline_cpu_ram_delta_mb,pipeline_param_count_total,pipeline_model_param_memory_mb_total,ratio_order
0,20260412_170120,mlp_inference,DNABert_zhihan1996,DNABERT-2-117M,DNABERT-2-117M,DNABERT-2-117M_w300,300,gencode,1_1_raw,5,...,18.129477,4.795247e+05,measured_profiler,18.903312,4.795525e+05,1529.661621,60.652344,117595907,448.592785,0
1,20260412_170120,mlp_inference,DNABert_zhihan1996,DNABERT-2-117M,DNABERT-2-117M,DNABERT-2-117M_w300,300,gtex,1_1_raw,5,...,18.129477,7.480585e+05,measured_profiler,29.277611,7.481018e+05,1530.482422,45.335938,117595907,448.592785,0
2,20260412_170120,mlp_inference,HyenaDNA_LongSafari,hyenadna-medium-160k-seqlen-hf,hyenadna-medium-160k-seqlen-hf,hyenadna-medium-160k-seqlen-hf_w300,300,gencode,1_1_raw,5,...,3.899910,1.031526e+05,measured_profiler,22.787776,1.031665e+05,2560.872070,-41.539062,6815747,26.000011,0
3,20260412_170120,mlp_inference,HyenaDNA_LongSafari,hyenadna-medium-160k-seqlen-hf,hyenadna-medium-160k-seqlen-hf,hyenadna-medium-160k-seqlen-hf_w300,300,gtex,1_1_raw,5,...,3.899910,1.609181e+05,measured_profiler,35.485477,1.609398e+05,2560.872070,17.125000,6815747,26.000011,0
4,20260412_170120,mlp_inference,HyenaDNA_LongSafari,hyenadna-small-32k-seqlen-hf,hyenadna-small-32k-seqlen-hf,hyenadna-small-32k-seqlen-hf_w300,300,gencode,1_1_raw,3,...,1.949955,5.157631e+04,measured_profiler,20.041319,5.159024e+04,1894.573242,440.363281,3542787,13.514660,0
5,20260412_170120,mlp_inference,HyenaDNA_LongSafari,hyenadna-small-32k-seqlen-hf,hyenadna-small-32k-seqlen-hf,hyenadna-small-32k-seqlen-hf_w300,300,gtex,1_1_raw,3,...,1.949955,8.045904e+04,measured_profiler,30.602077,8.048077e+04,1894.573242,46.152344,3542787,13.514660,0
6,20260412_170120,mlp_inference,NucleotideTransformer_InstaDeepAI,nucleotide-transformer-500m-human-ref,nucleotide-transformer-500m-human-ref,nucleotide-transformer-500m-human-ref_w300,300,gencode,1_1_raw,3,...,48.477406,1.282227e+06,measured_profiler,26.249124,1.282269e+06,4156.910645,38.046875,481227748,1835.738174,0
7,20260412_170120,mlp_inference,NucleotideTransformer_InstaDeepAI,nucleotide-transformer-500m-human-ref,nucleotide-transformer-500m-human-ref,nucleotide-transformer-500m-human-ref_w300,300,gtex,1_1_raw,3,...,48.477406,2.000275e+06,measured_profiler,41.289394,2.000340e+06,4156.910645,65.988281,481227748,1835.738174,0
8,20260412_170120,mlp_inference,NucleotideTransformer_InstaDeepAI,nucleotide-transformer-v2-100m-multi-species,nucleotide-transformer-v2-100m-multi-species,nucleotide-transformer-v2-100m-multi-species_w300,300,gencode,1_1_raw,2,...,7.189709,1.901678e+05,measured_profiler,11.095122,1.901887e+05,3163.130859,68.789062,72115556,275.099014,0
9,20260412_170120,mlp_inference,NucleotideTransformer_InstaDeepAI,nucleotide-transformer-v2-100m-multi-species,nucleotide-transformer-v2-100m-multi-species,nucleotide-transformer-v2-100m-multi-species_w300,300,gtex,1_1_raw,2,...,7.189709,2.966618e+05,measured_profiler,17.687529,2.966943e+05,3163.130859,23.976562,72115556,275.099014,0


In [2]:
# Tong hop nhanh theo ratio/dataset (FM + MLP + Pipeline)
display_cols = [
    'ratio', 'dataset', 'family', 'experiment', 'window_size',
    'balanced_accuracy', 'mcc', 'f1_weighted',
    'fm_param_count', 'fm_model_param_memory_mb', 'fm_gpu_peak_memory_mb', 'fm_cpu_ram_delta_mb', 'fm_gflops_per_sample', 'fm_gflops_total', 'fm_running_time_seconds',
    'mlp_param_count', 'mlp_model_param_memory_mb', 'mlp_gpu_peak_memory_mb', 'mlp_cpu_ram_delta_mb', 'mlp_gflops_per_sample', 'mlp_gflops_total', 'mlp_running_time_seconds',
    'pipeline_param_count_total', 'pipeline_model_param_memory_mb_total',
    'pipeline_gflops_total', 'pipeline_gpu_peak_memory_mb', 'pipeline_cpu_ram_delta_mb', 'pipeline_running_time_seconds',
    'n_samples', 'model_inference_seconds_total'
    ]

summary_df = results_df[display_cols].copy()
summary_df = summary_df.sort_values(['ratio', 'dataset', 'window_size', 'family', 'experiment']).reset_index(drop=True)

print('Summary rows:', len(summary_df))
summary_df.head(30)

Summary rows: 290


,ratio,dataset,family,experiment,window_size,balanced_accuracy,mcc,f1_weighted,fm_param_count,fm_model_param_memory_mb,...,mlp_gflops_total,mlp_running_time_seconds,pipeline_param_count_total,pipeline_model_param_memory_mb_total,pipeline_gflops_total,pipeline_gpu_peak_memory_mb,pipeline_cpu_ram_delta_mb,pipeline_running_time_seconds,n_samples,model_inference_seconds_total
0,1_1_10,gencode,DNABert_zhihan1996,DNABERT-2-117M_w300,300,0.327307,-0.009162,0.688535,117068544,446.581055,...,11.299094,0.006425,117595907,448.592785,1.949394e+05,1478.887695,-5.164062,7.763470,10752,0.075866
1,1_1_10,gencode,HyenaDNA_LongSafari,hyenadna-medium-160k-seqlen-hf_w300,300,0.800521,0.456605,0.745772,6550528,24.988281,...,5.661949,0.003559,6815747,26.000011,4.193749e+04,2560.872070,16.250000,9.382492,10752,0.050889
2,1_1_10,gencode,HyenaDNA_LongSafari,hyenadna-small-32k-seqlen-hf_w300,300,0.778497,0.404754,0.681473,3277568,12.502930,...,5.661949,0.003581,3542787,13.514660,2.097158e+04,1894.573242,25.750000,7.992528,10752,0.056884
3,1_1_10,gencode,NucleotideTransformer_InstaDeepAI,nucleotide-transformer-500m-human-ref_w300,300,0.904725,0.674920,0.879099,480438241,1832.726444,...,16.936239,0.006997,481227748,1835.738174,5.212460e+05,4156.910645,-25.433594,10.677385,10752,0.107271
4,1_1_10,gencode,NucleotideTransformer_InstaDeepAI,nucleotide-transformer-v2-100m-multi-species_w300,300,0.417187,0.089684,0.434864,71719265,273.587284,...,8.480522,0.003855,72115556,275.099014,7.731223e+04,3163.130859,7.152344,4.435705,10752,0.057807
5,1_1_10,gencode,NucleotideTransformer_InstaDeepAI,nucleotide-transformer-v2-250m-multi-species_w300,300,0.452307,0.125394,0.683820,173855617,663.206547,...,11.299094,0.004788,174382980,665.218277,1.886052e+05,4242.734375,-12.500000,6.906270,10752,0.073835
6,1_1_10,gencode,DNABert_zhihan1996,DNABERT-2-117M_w600,600,0.331799,-0.001593,0.548820,117068544,446.581055,...,11.236041,0.004691,117595907,448.592785,3.824355e+05,1514.234863,5.937500,15.096788,10692,0.074748
7,1_1_10,gencode,HyenaDNA_LongSafari,hyenadna-medium-160k-seqlen-hf_w600,600,0.550879,0.283022,0.757092,6550528,24.988281,...,5.630354,0.003622,6815747,26.000011,8.340127e+04,5820.423340,0.281250,18.240660,10692,0.050614
8,1_1_10,gencode,HyenaDNA_LongSafari,hyenadna-small-32k-seqlen-hf_w600,600,0.750767,0.379157,0.660647,3277568,12.502930,...,5.630354,0.003329,3542787,13.514660,4.170345e+04,5213.310059,-15.738281,15.791332,10692,0.054749
9,1_1_10,gencode,NucleotideTransformer_InstaDeepAI,nucleotide-transformer-500m-human-ref_w600,600,0.909877,0.703072,0.893825,480438241,1832.726444,...,16.841728,0.006559,481227748,1835.738174,1.033121e+06,5091.003906,-1.082031,20.495339,10692,0.098238
